# 8. Revisione agentica di un contratto

Analizza clausole, importi e firme con il workflow agentic.

## 1. Import e configurazione

Carica librerie, variabili di ambiente, schema e documento.

In [ ]:
import json
import os
from pathlib import Path
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import ContentAnalyzer
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

In [ ]:
load_dotenv(r"..\.env", override=True)


In [ ]:
document_bytes = Path(r"..\file\esercizio8.pdf").read_bytes()


In [ ]:
# Legge la definizione JSON, non il risultato di un'analisi.
definizione = json.loads(
    Path(r"..\modelli\contratto_agentic.json").read_text(encoding="utf-8")
)


## 2. Analisi agentica

Crea l'analyzer agentic, analizza il contratto e rimuove l'analyzer.

In [ ]:
credential = DefaultAzureCredential()
client = ContentUnderstandingClient(
    endpoint=os.environ["FOUNDRY_SERVICES_ENDPOINT"],
    credential=credential,
)


In [ ]:
# Crea l'analyzer dal JSON; allow_replace aggiorna l'ID del laboratorio.
try:
    client.begin_create_analyzer(
        analyzer_id="contratto_agentic",
        resource=ContentAnalyzer(definizione),
        allow_replace=True,
    ).result()
except Exception:
    client.close()
    credential.close()
    raise


In [ ]:
# Analizza i byte del PDF con l'analyzer indicato e attende il risultato.
try:
    result = client.begin_analyze_binary(
        analyzer_id="contratto_agentic",
        binary_input=document_bytes,
    ).result()
finally:
    try:
        # Elimina l'analyzer temporaneo identificato da analyzer_id.
        client.delete_analyzer(analyzer_id="contratto_agentic")
    finally:
        client.close()
        credential.close()


## 3. Risultato

Mostra calcoli, verifiche contrattuali ed esito motivato.

In [ ]:
fields = result.as_dict()["contents"][0]["fields"]
print(json.dumps(fields, ensure_ascii=False, indent=2))